문제 4.
인터파크 티켓 예매 페이지에서 인기 공연 데이터를 수집하고 공연 카테고리별 소비 분석
https://nol.interpark.com/ticket
1) 공연명, 날짜, 장소, 가격
2) MySQL 저장
3) SQL로 분석해야 하는 내용
장르별 평균 가격과 최저·최고 가격
0일 이내 시작하는 공연 중 가격 상위 5개
공연 장소별 등록된 공연 수 순위

In [16]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import time
import mysql.connector

# ==============================
# 1. MySQL 연결 설정
# ==============================
conn = mysql.connector.connect(
    host="localhost",        # MySQL 서버 주소
    user="root",             # MySQL 계정
    password="dnwhdgus9#"   # MySQL 비밀번호
)
cursor = conn.cursor()

# DB 생성 (없으면 새로 생성)
cursor.execute("CREATE DATABASE IF NOT EXISTS interpark1 CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;")
conn.database = "interpark1"

# 테이블 생성 (중복 방지 위해 UNIQUE KEY 설정)
cursor.execute("""
CREATE TABLE IF NOT EXISTS performances (
    id INT AUTO_INCREMENT PRIMARY KEY,
    genre VARCHAR(50),
    title VARCHAR(100),
    date VARCHAR(100),
    place VARCHAR(100),
    price VARCHAR(100),
    UNIQUE KEY unique_performance (title, date, place)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
""")
conn.commit()


# ==============================
# 2. DB 저장 함수 (중복 무시)
# ==============================
def save_to_db(genre, title, date, place, price):
    sql = """
    INSERT IGNORE INTO performances (genre, title, date, place, price)
    VALUES (%s, %s, %s, %s, %s)
    """
    cursor.execute(sql, (genre, title, date, place, price))
    conn.commit()


# ==============================
# 3. Selenium 크롤링 설정
# ==============================
service = Service(ChromeDriverManager().install())

options = Options()
options.add_argument("--window-size=1920x1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

# 장르별 파라미터 리스트
genres = ["MUSICAL", "CONCERT", "EXHIBIT", "CLASSIC", "KIDS", "DRAMA", "LEISURE"]

# ==============================
# 4. 크롤링 실행
# ==============================
for genre in genres:
    url = f"https://tickets.interpark.com/contents/ranking?genre={genre}"
    driver.get(url)
    time.sleep(3)

    print(f"\n\n========== {genre} 장르 공연 ==========")

    idx = 0
    while idx < 10:
        try:
            buttons = driver.find_elements(By.CSS_SELECTOR, "div.responsive-ranking-list_imageWrap__ROlYW")
            if idx >= len(buttons):
                break

            # 클릭
            driver.execute_script("arguments[0].click();", buttons[idx])
            time.sleep(3)

            print(f"\n----- {genre} {idx+1}번째 공연 -----")

            try:
                title = driver.find_element(By.CSS_SELECTOR, "h2.prdTitle").text.strip()
            except NoSuchElementException:
                title = "공연명 없음"

            try:
                date = driver.find_element(By.CSS_SELECTOR, "p.infoText").text.strip()
            except NoSuchElementException:
                date = "날짜 정보 없음"

            try:
                place = driver.find_element(By.CSS_SELECTOR, "a.infoBtn").text.strip()
            except NoSuchElementException:
                place = "장소 정보 없음"

            try:
                prices = driver.find_element(By.CSS_SELECTOR, "span.price").text.strip().replace("원", "")
            except NoSuchElementException:
                prices = "가격 정보 없음"

            print("공연명:", title)
            print("날짜:", date)
            print("장소:", place)
            print("가격:", prices)

            # ✅ DB 저장 (중복 무시)
            save_to_db(genre, title, date, place, prices)

            # 뒤로가기
            driver.back()
            time.sleep(3)

            idx += 1

        except StaleElementReferenceException:
            print(f"{idx+1}번째 요소 stale 참조 발생, 재시도")
            time.sleep(1)
            continue

# ==============================
# 5. 종료 처리
# ==============================
driver.quit()
cursor.close()
conn.close()



========== MUSICAL 장르 공연 ==========

----- MUSICAL 1번째 공연 -----
공연명: 뮤지컬 〈설공찬〉
날짜: 2025.09.09 ~2025.10.26
장소: KT＆G 상상마당 대치아트홀(자세히)
가격: 77,000

----- MUSICAL 2번째 공연 -----
공연명: 뮤지컬 〈멤피스〉
날짜: 2025.06.17 ~2025.09.21
장소: 충무아트센터 대극장(자세히)
가격: 170,000

----- MUSICAL 3번째 공연 -----
공연명: 뮤지컬 〈위키드〉 내한 공연(WICKED The Musical)
날짜: 2025.07.12 ~2025.10.26
장소: 블루스퀘어 신한카드홀(자세히)
가격: 190,000

----- MUSICAL 4번째 공연 -----
공연명: 뮤지컬 〈리틀잭〉
날짜: 2025.06.28 ~2025.09.21
장소: 예스24아트원 2관(자세히)
가격: 66,000

----- MUSICAL 5번째 공연 -----
공연명: 뮤지컬 〈노트르담 드 파리〉 프렌치 오리지널 내한공연 한국투어 20주년
날짜: 2025.09.03 ~2025.09.27
장소: 세종문화회관 대극장(자세히)
가격: 가격 정보 없음

----- MUSICAL 6번째 공연 -----
공연명: 뮤지컬 〈브로드웨이 42번가〉
날짜: 2025.07.10 ~2025.09.14
장소: 샤롯데씨어터(자세히)
가격: 160,000

----- MUSICAL 7번째 공연 -----
공연명: 공연명 없음
날짜: 날짜 정보 없음
장소: 장소 정보 없음
가격: 가격 정보 없음


========== CONCERT 장르 공연 ==========

----- CONCERT 1번째 공연 -----
공연명: 2025 LEE MINHYUK (HUTA) CONCERT 〈HOOK - WHO : KING〉
날짜: 2025.09.13 ~2025.09.14
장소: 명화라이브홀(자세히)
가격: 154,000

----- CONCERT 2번째 공연 -----
공